In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


In [2]:
X_train = np.load("../artifacts/data/X_train.npy")
y_train = np.load("../artifacts/data/y_train.npy")


In [3]:
#aqui já entro com aprendizado constrativo/métrica
#saio de 'classificar' para 'diferenciar'
#saio da classificação binária para um um espaço latente
#importância: modelo se torna robusto à variações de ataque; detecta anomalias (zero-day)
#acelero o treino otimizando o sorteiro com o pos e neg indices

class TripletDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.pos_indices = {}
        self.neg_indices = {}

        for label in np.unique(y):
            self.pos_indices[label] = np.where(y == label)[0] 
            self.neg_indices[label] = np.where(y != label)[0]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        anchor = self.X[idx]
        label = self.y[idx]

        pos_idx = np.random.choice(self.pos_indices[label]) #introduzo variabilidade necessária para que o modelo veja combinações
        neg_idx = np.random.choice(self.neg_indices[label]) #infinitas de tripas, tornando-o mais difícil de ser enganado

        positive = self.X[pos_idx]
        negative = self.X[neg_idx]

        return (
            torch.tensor(anchor, dtype=torch.float32),
            torch.tensor(positive, dtype=torch.float32),
            torch.tensor(negative, dtype=torch.float32),
        )


In [4]:
#garanto estabilidade com uma hipersfera, forçando o modelo a aprender a angulação e 
#direção das 194 características, e não simplesmente aumentando infinitamente a distância entre os pontos
#64 é a asssinatura digital
#as 194 dimensões são comprimidas em 64 coordenadas essenciais
#após o treino, os ataques estarão em uma região da esfera, enquanto o normal estarão do lado oposto

class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, p=2, dim=1)


In [5]:
dataset = TripletDataset(X_train, y_train)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True
)


In [6]:
#aqui não basta o ataque estar longe do tráfego normal, ele deve estar a uma margem de segurança mínima para que o modelo para de se preocupar com ele

device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = Encoder(X_train.shape[1]).to(device)

criterion = nn.TripletMarginLoss(margin=1.0)

optimizer = optim.Adam(encoder.parameters(), lr=0.001)


In [7]:
epochs = 15

for epoch in range(epochs):
    encoder.train()
    total_loss = 0

    for anchor, positive, negative in loader:
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()

        #gero vetores de 64 posições
        z_a = encoder(anchor)
        z_p = encoder(positive)
        z_n = encoder(negative)

        #calculo a perda baseada na distância relativa
        loss = criterion(z_a, z_p, z_n)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")


Epoch 1/15 - Loss: 0.2340
Epoch 2/15 - Loss: 0.2022
Epoch 3/15 - Loss: 0.1941
Epoch 4/15 - Loss: 0.1904
Epoch 5/15 - Loss: 0.1875
Epoch 6/15 - Loss: 0.1853
Epoch 7/15 - Loss: 0.1842
Epoch 8/15 - Loss: 0.1829
Epoch 9/15 - Loss: 0.1806
Epoch 10/15 - Loss: 0.1809
Epoch 11/15 - Loss: 0.1783
Epoch 12/15 - Loss: 0.1784
Epoch 13/15 - Loss: 0.1786
Epoch 14/15 - Loss: 0.1763
Epoch 15/15 - Loss: 0.1763


In [8]:
#salvo o modelo
torch.save(
    encoder.state_dict(),
    "../artifacts/models/encoder_contrastive.pth"
)
